# 🤖 CCE Multi-Agent Exploration — Production Experiments (FIXED)

**Status:** ✅ Fixed for End-Term Final Report (Nov 22, 2026)

**Key fixes vs. the original notebook:**

1. **Reward shaping** (`cce/envs/grid_world.py`): boosted coverage reward ×5, reduced overlap/time penalties ×0.1.
2. **λ non-negativity** (`cce/comm/budget.py`): confirmed `clamp_(min=0.0, max=max_lambda)` is active and added stronger PID damping after warmup.
3. **50-update smoke gate**: full sweep only starts if a quick smoke run reaches coverage > 0.75.
4. **Expanded β sweep**: `[0.0, 0.25, 0.5, 0.75, 1.0]` — 5 points for a proper Pareto frontier.
5. **VQ logging verification**: diagnostic cell prints `vq_loss` and `perplexity`.
6. **Baseline eval**: all β checkpoints are evaluated with matching config flags.
7. **Diagnostic audit**: prints the actual current source of the reward, budget, and eval modules.

---


In [1]:
import os, subprocess, shutil, json

REPO_URL = 'https://github.com/Govindu1729/cce-multi-agent-exploration.git'
BRANCH = 'main'
TARGET_DIR = '/kaggle/working/cce_project'
OUTPUT_DIR = '/kaggle/working/experiment_output'

for d in [TARGET_DIR, OUTPUT_DIR]:
    if os.path.exists(d): shutil.rmtree(d)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('🚀 Cloning ' + BRANCH + '...')
subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, TARGET_DIR], check=True)
os.chdir(TARGET_DIR)
print('📍 Working dir: ' + os.getcwd())
print('📜 Latest commit:')
print(subprocess.check_output(['git', 'log', '-1', '--oneline'], text=True))

print('📦 Installing deps...')
subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)

print('\n🔬 Verifying setup...')
subprocess.run(['python', 'verify_setup.py'])
print('\n✅ Setup complete!')


🚀 Cloning main...


Cloning into '/kaggle/working/cce_project'...


📍 Working dir: /kaggle/working/cce_project
📜 Latest commit:
0776c3c feat: update fai-project notebook with production experiments and baseline comparisons

📦 Installing deps...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.9/168.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 649.8/649.8 kB 23.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 8.4 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.18.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.



🔬 Verifying setup...
Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
✓ PyTorch 2.11.0+cu128
  CUDA available: True
✓ NumPy 2.1.3
✓ Gymnasium 1.2.0
✓ cce.envs.grid_world
✓ cce.algo.mappo
✓ cce.comm.vq
✓ cce.comm.budget

✓ All checks passed! Ready to train.

✅ Setup complete!


In [2]:
# Python 3.13 syntax fix: `lambda` is reserved; rename to `lam`.
import os, sys
os.chdir('/kaggle/working/cce_project')

for test_file, replacements in [
    ('tests/test_budget.py', [('bc.lambda', 'bc.lam')]),
    ('cce/comm/budget.py',   [('self.lambda', 'self.lam')]),
]:
    if not os.path.exists(test_file): continue
    with open(test_file, 'r') as f: code = f.read()
    changed = False
    for old, new in replacements:
        if old in code:
            code = code.replace(old, new)
            changed = True
    if changed:
        with open(test_file, 'w') as f: f.write(code)
        print('✅ Patched ' + str(replacements) + ' in ' + test_file)

os.system('find . -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null')
os.environ['PYTHONPATH'] = '/kaggle/working/cce_project:' + os.environ.get('PYTHONPATH', '')
sys.path.insert(0, '/kaggle/working/cce_project')
print('🧪 Running tests...')
os.system('PYTHONPATH=/kaggle/working/cce_project pytest tests/ -q --tb=short')
print('\n✅ All tests should pass before proceeding.')


🧪 Running tests...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working/cce_project
configfile: pyproject.toml
plugins: hydra-core-1.3.8, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 27 items

tests/test_budget.py ......                                              [ 22%]
tests/test_buffer.py ...                                                 [ 33%]
tests/test_grid_world.py .....                                           [ 51%]
tests/test_mapgen.py .....                                               [ 70%]
tests/test_mappo.py ...                                                  [ 81%]
tests/test_vq.py .....                                                   [100%]

============================== 27 passed in 7.52s ==============================

✅ All tests should pass before proceeding.


In [11]:
import subprocess, sys, os
import pandas as pd

# Ensure we are in the correct repo root directory
if os.path.exists("/kaggle/working/cce_project"):
    os.chdir("/kaggle/working/cce_project")
elif os.path.exists("cce-multi-agent-exploration") and not os.path.exists("cce"):
    os.chdir("cce-multi-agent-exploration")
    
print(f"Working directory: {os.getcwd()}\n")

print("="*70)
print("=== 1. PYTEST DETAILS (Identifying the failing test assertions) ===")
print("="*70)
res = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True,
)
print(res.stdout)
if res.stderr: print("STDERR:", res.stderr)

print("\n" + "="*70)
print("=== 2. VERIFYING FRONTIER BASELINE (100 steps, 20x20 grid) ===")
print("="*70)
res_f = subprocess.run(
    [sys.executable, "scripts/evaluate.py", "--baseline", "frontier",
     "--grid_size", "20", "--num_agents", "4", "--fov_size", "5",
     "--max_steps", "100", "--num_episodes", "5", "--seed", "42"],
    capture_output=True, text=True,
)
print(res_f.stdout)
if res_f.stderr: print("STDERR:", res_f.stderr)

print("\n" + "="*70)
print("=== 3. RUNNING 50-UPDATE SMOKE TEST (β=0.25) ===")
print("="*70)
res_s = subprocess.run(
    [sys.executable, "train.py", "--beta", "0.25", "--num_envs", "32",
     "--num_updates", "50", "--grid_size", "20", "--num_agents", "4",
     "--fov_size", "5", "--max_steps", "100", "--actor_hidden", "64",
     "--critic_hidden", "128", "--n_codes", "16", "--code_dim", "4",
     "--n_slots", "2", "--lr_actor", "1e-3", "--run_dir", "runs/smoke",
     "--run_name", "smoke_test"],
    capture_output=True, text=True,
)
# Print the last 20 lines of training output
print("\n".join(res_s.stdout.splitlines()[-20:]))
if res_s.returncode != 0:
    print("SMOKE TEST STDERR (last 1000 chars):", res_s.stderr[-1000:])

# Parse and print summary
if os.path.exists("runs/smoke/metrics.jsonl"):
    df = pd.read_json("runs/smoke/metrics.jsonl", lines=True)
    if len(df) > 0:
        first10 = df["coverage"].head(10).mean()
        last10 = df["coverage"].tail(10).mean()
        peak = df["coverage"].max()
        print(f"\n✅ Smoke Summary -> Updates: {len(df)} | First-10 mean: {first10:.3f} | Last-10 mean: {last10:.3f} | Peak: {peak:.3f}")
        if peak >= 0.75 and last10 >= first10:
            print("🎉 SMOKE TEST PASSED! You are clear to run the full production sweep.")
    else:
        print("\n⚠️ metrics.jsonl is empty.")
else:
    print("\n⚠️ metrics.jsonl not found. Smoke test may have crashed before logging.")

Working directory: /kaggle/working/cce_project

=== 1. PYTEST DETAILS (Identifying the failing test assertions) ===
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/cce_project
configfile: pyproject.toml
plugins: hydra-core-1.3.8, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collecting ... collected 27 items

tests/test_budget.py::test_budget_controller_policy_mode PASSED          [  3%]
tests/test_budget.py::test_budget_controller_utility_mode PASSED         [  7%]
tests/test_budget.py::test_budget_controller_update PASSED               [ 11%]
tests/test_budget.py::test_budget_controller_converges PASSED            [ 14%]
tests/test_budget.py::test_budget_controller_decide_send_utility PASSED  [ 18%]
tests/test_budget.py::test_hard_budget_enforcement PASSED                [ 22%]
tests/test_buffer.py::test_buffer_insert PAS

In [12]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

# Ensure we are in the repo root
if os.path.exists("cce-multi-agent-exploration") and not os.path.exists("cce"):
    os.chdir("cce-multi-agent-exploration")
print(f"Working directory: {os.getcwd()}\n")

print("="*70)
print("=== 1. FIXING mappo.py (MultiActionDist API Bug) ===")
print("="*70)
mappo_path = Path("cce/algo/mappo.py")
src = mappo_path.read_text()

old_block = """            # Train movement action with PPO advantage; budget controller regulates send gate
            log_probs_move_new = dist.move_dist.log_prob(batch["actions_move"])
            log_probs_send_new = dist.send_dist.log_prob(batch["actions_send"].float())
            log_probs_new = log_probs_move_new + log_probs_send_new

            move_entropy = dist.move_dist.entropy().mean()
            send_entropy = msg_info["send_entropy"].mean()
            entropy = move_entropy + 0.1 * send_entropy"""

new_block = """            # Train actions with PPO advantage (using unified MultiActionDist API)
            log_probs_new = dist.log_prob(batch["actions_move"], batch["actions_send"])
            
            # Entropy bonus
            entropy = dist.entropy().mean()
            if "send_entropy" in msg_info and hasattr(msg_info["send_entropy"], "mean"):
                entropy = entropy + 0.1 * msg_info["send_entropy"].mean()"""

if old_block in src:
    src = src.replace(old_block, new_block)
    mappo_path.write_text(src)
    print("✅ Successfully patched mappo.py!")
else:
    print("⚠️ Exact block not found. Attempting regex fallback...")
    import re
    pattern = r'# Train movement action with PPO advantage.*?entropy = move_entropy \+ 0\.1 \* send_entropy'
    if re.search(pattern, src, re.DOTALL):
        src = re.sub(pattern, new_block.strip(), src, flags=re.DOTALL)
        mappo_path.write_text(src)
        print("✅ Patched via regex fallback!")
    else:
        print("❌ Could not patch. The code might already be fixed or changed.")

os.system('find . -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null')

print("\n" + "="*70)
print("=== 2. RE-RUNNING 50-UPDATE SMOKE TEST (β=0.25) ===")
print("="*70)
res_s = subprocess.run(
    [sys.executable, "train.py", "--beta", "0.25", "--num_envs", "32",
     "--num_updates", "50", "--grid_size", "20", "--num_agents", "4",
     "--fov_size", "5", "--max_steps", "100", "--actor_hidden", "64",
     "--critic_hidden", "128", "--n_codes", "16", "--code_dim", "4",
     "--n_slots", "2", "--lr_actor", "1e-3", "--run_dir", "runs/smoke",
     "--run_name", "smoke_test"],
    capture_output=True, text=True,
)

# Print the last 25 lines of training output
print("\n".join(res_s.stdout.splitlines()[-25:]))
if res_s.returncode != 0:
    print("SMOKE TEST STDERR (last 1000 chars):", res_s.stderr[-1000:])

# Parse and print summary
if os.path.exists("runs/smoke/metrics.jsonl"):
    df = pd.read_json("runs/smoke/metrics.jsonl", lines=True)
    if len(df) > 0:
        first10 = df["coverage"].head(10).mean()
        last10 = df["coverage"].tail(10).mean()
        peak = df["coverage"].max()
        print(f"\n✅ Smoke Summary -> Updates: {len(df)} | First-10 mean: {first10:.3f} | Last-10 mean: {last10:.3f} | Peak: {peak:.3f}")
        if peak >= 0.75 and last10 >= first10:
            print("🎉 SMOKE TEST PASSED! Coverage is climbing. You are clear to run the full production sweep.")
        else:
            print("⚠️ Smoke test ran, but coverage didn't meet the strict gate. Check the curve.")

Working directory: /kaggle/working/cce_project

=== 1. FIXING mappo.py (MultiActionDist API Bug) ===
✅ Successfully patched mappo.py!

=== 2. RE-RUNNING 50-UPDATE SMOKE TEST (β=0.25) ===
Training with β=0.25, 32 envs, seed=0
Device: cuda
Parameters: actor=17,933, critic=82,305
SMOKE TEST STDERR (last 1000 chars): Traceback (most recent call last):
  File "/kaggle/working/cce_project/train.py", line 164, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/cce_project/train.py", line 125, in main
    stats = agent.train_step(logger if update % args.log_interval == 0 else None)
  File "/kaggle/working/cce_project/cce/algo/mappo.py", line 355, in train_step
    update_stats = self.update()
  File "/kaggle/working/cce_project/cce/algo/mappo.py", line 339, in update
    total_entropy += move_entropy.item()
                     ^^^^^^^^^^^^
NameError: name 'move_entropy' is not defined. Did you mean: 'total_entropy'?


✅ Smoke Summary -> Updates: 50 | First-10 mean: 0.715 | Last-10 mean

In [13]:
import os, sys, subprocess
from pathlib import Path
import pandas as pd

print("=== 1. FIXING mappo.py (move_entropy NameError) ===")
mappo_path = Path("cce/algo/mappo.py")
src = mappo_path.read_text()

old_line = "            total_entropy += move_entropy.item()"
new_line = "            total_entropy += entropy.item()"

if old_line in src:
    src = src.replace(old_line, new_line)
    mappo_path.write_text(src)
    print("✅ Successfully patched mappo.py!")
else:
    print("⚠️ Could not find the exact line. Attempting regex...")
    import re
    src = re.sub(r'total_entropy \+= move_entropy\.item\(\)', 'total_entropy += entropy.item()', src)
    mappo_path.write_text(src)
    print("✅ Patched via regex fallback!")

os.system('find . -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null')

print("\n=== 2. RE-RUNNING 50-UPDATE SMOKE TEST (β=0.25) ===")
res_s = subprocess.run(
    [sys.executable, "train.py", "--beta", "0.25", "--num_envs", "32",
     "--num_updates", "50", "--grid_size", "20", "--num_agents", "4",
     "--fov_size", "5", "--max_steps", "100", "--actor_hidden", "64",
     "--critic_hidden", "128", "--n_codes", "16", "--code_dim", "4",
     "--n_slots", "2", "--lr_actor", "1e-3", "--run_dir", "runs/smoke",
     "--run_name", "smoke_test"],
    capture_output=True, text=True,
)

# Print the last 25 lines of training output
print("\n".join(res_s.stdout.splitlines()[-25:]))
if res_s.returncode != 0:
    print("SMOKE TEST STDERR (last 1000 chars):", res_s.stderr[-1000:])

# Parse and print summary
if os.path.exists("runs/smoke/metrics.jsonl"):
    df = pd.read_json("runs/smoke/metrics.jsonl", lines=True)
    if len(df) > 0:
        first10 = df["coverage"].head(10).mean()
        last10 = df["coverage"].tail(10).mean()
        peak = df["coverage"].max()
        print(f"\n✅ Smoke Summary -> Updates: {len(df)} | First-10 mean: {first10:.3f} | Last-10 mean: {last10:.3f} | Peak: {peak:.3f}")
        if peak >= 0.75 and last10 >= first10:
            print("🎉 SMOKE TEST PASSED! Coverage is climbing. You are clear to run the full production sweep.")
        else:
            print("⚠️ Smoke test ran, but coverage didn't meet the strict gate. Check the curve.")

=== 1. FIXING mappo.py (move_entropy NameError) ===
✅ Successfully patched mappo.py!

=== 2. RE-RUNNING 50-UPDATE SMOKE TEST (β=0.25) ===
[27/50] cov=0.948 t95=64.9 β=0.434 λ=3.642 actor_loss=-0.0188 entropy=1.644
[28/50] cov=0.950 t95=66.2 β=0.434 λ=3.762 actor_loss=-0.0184 entropy=1.640
[29/50] cov=0.956 t95=58.6 β=0.440 λ=3.885 actor_loss=-0.0196 entropy=1.625
[30/50] cov=0.949 t95=59.6 β=0.438 λ=4.007 actor_loss=-0.0181 entropy=1.625
[31/50] cov=0.950 t95=62.8 β=0.432 λ=4.126 actor_loss=-0.0200 entropy=1.589
[32/50] cov=0.956 t95=55.3 β=0.445 λ=4.253 actor_loss=-0.0174 entropy=1.588
[33/50] cov=0.955 t95=66.2 β=0.430 λ=4.370 actor_loss=-0.0189 entropy=1.570
[34/50] cov=0.962 t95=65.1 β=0.435 λ=4.490 actor_loss=-0.0185 entropy=1.573
[35/50] cov=0.931 t95=64.0 β=0.440 λ=4.613 actor_loss=-0.0192 entropy=1.551
[36/50] cov=0.954 t95=62.3 β=0.424 λ=4.726 actor_loss=-0.0187 entropy=1.551
[37/50] cov=0.950 t95=60.2 β=0.458 λ=4.861 actor_loss=-0.0187 entropy=1.522
[38/50] cov=0.935 t95=64.7

## 🔍 Diagnostic Audit — What code are we actually running?

Prints the reward weights, λ clamp, and evaluate CLI so we can verify the fixes are live.


In [3]:
import re

print('=== 1. Reward shaping in cce/envs/grid_world.py ===')
with open('cce/envs/grid_world.py') as f: src = f.read()
for w in ['w_coverage', 'w_overlap', 'w_time', 'w_comm', 'success_bonus']:
    m = re.search(rf'{w}:\s*float\s*=\s*([0-9.\-]+)', src)
    print('  ' + w.ljust(16) + ' = ' + (m.group(1) if m else '???'))

print('\n=== 2. λ clamp in cce/comm/budget.py ===')
with open('cce/comm/budget.py') as f: src = f.read()
m = re.search(r'lam\.clamp_\([^)]+\)', src)
print('  ' + (m.group(0) if m else '❌ NO CLAMP — λ can go negative!'))
m = re.search(r'max_lambda:\s*float\s*=\s*([0-9.]+)', src)
print('  max_lambda default = ' + (m.group(1) if m else '???'))

print('\n=== 3. Bootcamp warmup horizon in cce/algo/mappo.py ===')
with open('cce/algo/mappo.py') as f: src = f.read()
m = re.search(r'warmup\s*=\s*self\._update_idx\s*<=\s*(\d+)', src)
print('  warmup horizon = ' + (m.group(1) if m else '???') + ' updates')

print('\n=== 4. scripts/evaluate.py CLI (if present) ===')
if os.path.exists('scripts/evaluate.py'):
    with open('scripts/evaluate.py') as f: src = f.read()
    for flag in ['--baseline', '--grid_size', '--num_agents', '--max_steps',
                 '--fov_size', '--record', '--checkpoint']:
        found = flag in src
        print('  ' + flag.ljust(15) + ': ' + ('✅' if found else '❌'))
else:
    print('  ❌ scripts/evaluate.py does not exist')


=== 1. Reward shaping in cce/envs/grid_world.py ===
  w_coverage       = 1.0
  w_overlap        = 0.1
  w_time           = 0.01
  w_comm           = 0.05
  success_bonus    = 2.0

=== 2. λ clamp in cce/comm/budget.py ===
  lam.clamp_(min=0.0, max=self.max_lambda)
  max_lambda default = 10.0

=== 3. Bootcamp warmup horizon in cce/algo/mappo.py ===
  warmup horizon = 120 updates

=== 4. scripts/evaluate.py CLI (if present) ===
  --baseline     : ✅
  --grid_size    : ✅
  --num_agents   : ✅
  --max_steps    : ✅
  --fov_size     : ✅
  --record       : ✅
  --checkpoint   : ✅


## 🛠️ Patch: reward shaping

The original reward (`w_coverage=1.0, w_overlap=0.1, w_time=0.01`) penalises moving more than it rewards new cells. Under this shaping, PPO learns to **stop exploring** — which is exactly what the original notebook's training curves show (coverage drifting down from 0.72 to 0.58).

Fix: boost the coverage reward 5× and shrink the per-step penalties so that exploring new cells dominates.


In [4]:
with open('cce/envs/grid_world.py', 'r') as f: src = f.read()

patches = [
    ('w_coverage: float = 1.0',    'w_coverage: float = 5.0'),
    ('w_overlap: float = 0.1',     'w_overlap: float = 0.02'),
    ('w_time: float = 0.01',       'w_time: float = 0.001'),
    ('w_comm: float = 0.05',       'w_comm: float = 0.01'),
    ('success_bonus: float = 2.0', 'success_bonus: float = 5.0'),
]
for old, new in patches:
    if old in src:
        src = src.replace(old, new)
        print('  ✅ ' + old + ' → ' + new)
    else:
        print('  ⚠️  pattern not found: ' + repr(old))

with open('cce/envs/grid_world.py', 'w') as f: f.write(src)
os.system('find . -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null')
print('\n✅ Reward shaping patched.')


  ✅ w_coverage: float = 1.0 → w_coverage: float = 5.0
  ✅ w_overlap: float = 0.1 → w_overlap: float = 0.02
  ✅ w_time: float = 0.01 → w_time: float = 0.001
  ✅ w_comm: float = 0.05 → w_comm: float = 0.01
  ✅ success_bonus: float = 2.0 → success_bonus: float = 5.0

✅ Reward shaping patched.


## 🛠️ Patch: λ damping after warmup

`budget.py` already clamps `λ ∈ [0, max_lambda]`. We additionally reduce the PID gains once warmup ends so that `λ` does not oscillate between 0 and 5+ as in the original run.


In [5]:
with open('cce/algo/mappo.py', 'r') as f: src = f.read()

old = '            else:\n                # Tame the PID controller once the real constraints kick in\n                _bc.pid_kp = 0.05\n                _bc.pid_ki = 0.005\n                _bc.pid_kd = 0.01'
new = '            else:\n                # Tame the PID controller once the real constraints kick in\n                _bc.pid_kp = 0.02\n                _bc.pid_ki = 0.002\n                _bc.pid_kd = 0.005\n                # Clamp λ to [0, 2.0] to prevent runaway oscillation\n                if hasattr(_bc, \'max_lambda\'):\n                    _bc.max_lambda = min(_bc.max_lambda, 2.0)'

if old in src:
    src = src.replace(old, new)
    with open('cce/algo/mappo.py', 'w') as f: f.write(src)
    print('✅ Patched PID damping in mappo.py')
else:
    print('⚠️  Pattern not found; inspect manually.')

os.system('find . -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null')


✅ Patched PID damping in mappo.py


0

## 🚦 Smoke Test Gate (50 updates, β=0.5)

If coverage doesn't climb above 0.75 in 50 updates, **stop here and fix the shaping** — don't waste an hour on a full sweep that will produce the same broken curves.


In [6]:
smoke_cmd = [
    'python', 'train.py',
    '--beta', '0.5', '--num_envs', '32', '--num_updates', '50',
    '--grid_size', '20', '--num_agents', '4', '--fov_size', '5', '--max_steps', '100',
    '--actor_hidden', '64', '--critic_hidden', '128',
    '--n_codes', '16', '--code_dim', '4', '--n_slots', '2',
    '--run_dir', 'runs/smoke', '--run_name', 'smoke',
]
subprocess.run(smoke_cmd)

with open('runs/smoke/metrics.jsonl') as f:
    records = [json.loads(l) for l in f]

covs = [r['coverage'] for r in records]
lams = [r['lambda'] for r in records]
betas = [r['actual_beta'] for r in records]
vq = [r.get('vq_loss', 0.0) for r in records]
plx = [r.get('perplexity', 0.0) for r in records]

print('\nSmoke coverage (first 5):  ' + str([round(c,3) for c in covs[:5]]))
print('Smoke coverage (last 5):   ' + str([round(c,3) for c in covs[-5:]]))
print('Smoke λ        (last 5):   ' + str([round(l,3) for l in lams[-5:]]))
print('Smoke actual β (last 5):   ' + str([round(b,3) for b in betas[-5:]]))
print('Smoke vq_loss  (last 5):   ' + str([round(v,4) for v in vq[-5:]]))
print('Smoke perplexity (last 5): ' + str([round(p,3) for p in plx[-5:]]))

peak = max(covs)
if peak >= 0.75:
    print('\n✅ SMOKE PASSED: peak coverage %.3f >= 0.75 — proceeding.' % peak)
else:
    raise RuntimeError('❌ SMOKE FAILED: peak coverage %.3f < 0.75. Do not run full sweep.' % peak)


Training with β=0.5, 32 envs, seed=0
Device: cuda
Parameters: actor=17,933, critic=82,305
[1/50] cov=0.705 t95=-1.0 β=0.494 λ=0.000 actor_loss=-0.0001 entropy=2.773
[2/50] cov=0.701 t95=-1.0 β=0.501 λ=0.001 actor_loss=-0.0000 entropy=2.773
[3/50] cov=0.713 t95=-1.0 β=0.490 λ=0.000 actor_loss=-0.0001 entropy=2.772
[4/50] cov=0.724 t95=-1.0 β=0.490 λ=0.000 actor_loss=-0.0001 entropy=2.772
[5/50] cov=0.722 t95=-1.0 β=0.497 λ=0.000 actor_loss=-0.0000 entropy=2.772
[6/50] cov=0.721 t95=-1.0 β=0.495 λ=0.000 actor_loss=-0.0001 entropy=2.771
[7/50] cov=0.723 t95=-1.0 β=0.490 λ=0.000 actor_loss=-0.0007 entropy=2.769
[8/50] cov=0.740 t95=-1.0 β=0.465 λ=0.000 actor_loss=-0.0019 entropy=2.753
[9/50] cov=0.692 t95=86.0 β=0.436 λ=0.000 actor_loss=-0.0005 entropy=2.737
[10/50] cov=0.711 t95=-1.0 β=0.406 λ=0.000 actor_loss=-0.0013 entropy=2.698
[11/50] cov=0.709 t95=-1.0 β=0.383 λ=0.000 actor_loss=-0.0014 entropy=2.678
[12/50] cov=0.720 t95=-1.0 β=0.384 λ=0.000 actor_loss=-0.0012 entropy=2.677
[13/50]

RuntimeError: ❌ SMOKE FAILED: peak coverage 0.740 < 0.75. Do not run full sweep.

## 🧪 Experiment 1: β-Sweep Production Training

Five β values — enough to draw a proper Pareto frontier.


In [ ]:
import os, subprocess, shutil

BETAS = [0.0, 0.25, 0.5, 0.75, 1.0]
NUM_UPDATES = 300
OUTPUT_DIR = '/kaggle/working/experiment_output'

for beta in BETAS:
    tag = str(beta).replace('.', '_')
    print('=' * 60)
    print(f'PRODUCTION RUN: β={beta}')
    print('=' * 60)

    cmd = [
        'python', 'train.py',
        '--beta', str(beta),
        '--num_envs', '32',
        '--num_updates', str(NUM_UPDATES),
        '--grid_size', '20',
        '--num_agents', '4',
        '--fov_size', '5',
        '--max_steps', '100',
        '--actor_hidden', '64',
        '--critic_hidden', '128',
        '--n_codes', '16',
        '--code_dim', '4',
        '--n_slots', '2',
        '--run_dir', f'runs/production/beta_{tag}',
        '--run_name', f'prod_beta_{tag}',
    ]
    subprocess.run(cmd)

    # Copy artifacts to output
    run_dir = f'runs/production/beta_{tag}'
    out_dir = f'{OUTPUT_DIR}/beta_{tag}'
    os.makedirs(out_dir, exist_ok=True)
    for f in ['metrics.jsonl', 'summary.json', 'args.json']:
        src = f'{run_dir}/{f}'
        if os.path.exists(src):
            shutil.copy(src, out_dir)

    ckpt_dir = f'{run_dir}/checkpoints'
    if os.path.exists(ckpt_dir):
        shutil.copytree(ckpt_dir, f'{out_dir}/checkpoints', dirs_exist_ok=True)

    print(f'\n💾 Artifacts saved to {out_dir}\n')

PRODUCTION RUN: β=0.0


## 📈 Experiment 2: Paper Figures

1. Coverage vs. Updates — learning curves for all 5 β values
2. Actual Bandwidth vs. Target Budget
3. λ (Dual Variable) — should stay in [0, 2.0]
4. Pareto Frontier — Bandwidth vs. Coverage


In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt, numpy as np

BETAS = [0.0, 0.25, 0.5, 0.75, 1.0]
OUTPUT_DIR = '/kaggle/working/experiment_output'
all_runs = []

for beta in BETAS:
    tag = str(beta).replace('.', '_')
    path = f'{OUTPUT_DIR}/beta_{tag}/metrics.jsonl'
    if not os.path.exists(path):
        print(f'⚠️ Missing {path}')
        continue
    with open(path) as f:
        for line in f:
            rec = json.loads(line)
            rec['target_beta'] = beta
            all_runs.append(rec)

df = pd.DataFrame(all_runs)
df['update'] = df.groupby('target_beta').cumcount() + 1
print(f'Loaded {len(df)} records across {df.target_beta.nunique()} runs')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Coverage curves
ax = axes[0, 0]
for beta in BETAS:
    sub = df[df.target_beta == beta]
    ax.plot(sub['update'], sub['coverage'], label=f'β={beta}')
ax.axhline(0.72, color='grey', ls='--', alpha=0.5, label='Random Walk Baseline (~0.72)')
ax.set_xlabel('Update'); ax.set_ylabel('Coverage')
ax.set_title('Coverage vs. Training'); ax.legend(); ax.grid(True)

# 2. Actual bandwidth
ax = axes[0, 1]
for beta in BETAS:
    sub = df[df.target_beta == beta]
    ax.plot(sub['update'], sub['actual_beta'], label=f'β={beta} (actual)')
    ax.axhline(y=beta, color='k', ls='--', alpha=0.3)
ax.set_xlabel('Update'); ax.set_ylabel('Actual β')
ax.set_title('Bandwidth vs. Target Budget'); ax.legend(); ax.grid(True)

# 3. λ dynamics
ax = axes[1, 0]
for beta in BETAS[1:]: # Skip 0.0 since lambda is trivially 0
    sub = df[df.target_beta == beta]
    ax.plot(sub['update'], sub['lambda'], label=f'β={beta}')
ax.set_xlabel('Update'); ax.set_ylabel('λ (dual)')
ax.set_title('Lagrangian Multiplier (λ ≥ 0)'); ax.legend(); ax.grid(True)

# 4. Pareto frontier
ax = axes[1, 1]
final = df.groupby('target_beta').last()
ax.scatter(final['actual_beta'], final['coverage'], s=100, zorder=5, color='C0')
for b, row in final.iterrows():
    ax.annotate(f'β={b}', (row.actual_beta, row.coverage),
                textcoords='offset points', xytext=(0, 10), ha='center')
ax.set_xlabel('Actual Bandwidth (β)'); ax.set_ylabel('Final Coverage')
ax.set_title('Pareto Frontier'); ax.grid(True)

plt.tight_layout()
fig_path = f'{OUTPUT_DIR}/paper_figures.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'💾 Figure saved: {fig_path}')

## 🆚 Experiment 3: Baseline Comparison

Evaluate **every** β checkpoint (not just β=0.5) and the Frontier baseline on the **same** grid / FOV / horizon so the numbers are comparable.


In [ ]:
import os, subprocess, json

OUTPUT_DIR = '/kaggle/working/experiment_output'
BETAS = [0.0, 0.25, 0.5, 0.75, 1.0]
EVAL_EPISODES = 20
SEED = 42
SHARED_FLAGS = [
    '--grid_size', '20', '--num_agents', '4', '--fov_size', '5',
    '--max_steps', '100', '--num_episodes', str(EVAL_EPISODES),
    '--seed', str(SEED),
]

# Frontier baseline
print("Evaluating Frontier Baseline...")
frontier_cmd = ['python', 'scripts/evaluate.py',
                '--baseline', 'frontier',
                '--output', f'{OUTPUT_DIR}/eval_frontier.json',
                '--record',
                '--record_output', f'{OUTPUT_DIR}/trajectory_frontier.pkl'] + SHARED_FLAGS
subprocess.run(frontier_cmd)

# MAPPO for every β
for beta in BETAS:
    tag = str(beta).replace('.', '_')
    run_dir = f'runs/production/beta_{tag}'
    ckpt_dir = f'{OUTPUT_DIR}/beta_{tag}/checkpoints'
    
    # Find the latest checkpoint or best_model.pt
    ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.endswith('.pt')]) if os.path.isdir(ckpt_dir) else []
    ckpt = ckpts[-1] if ckpts else 'best_model.pt'
    ckpt_path = f'{ckpt_dir}/{ckpt}'
    
    print(f"Evaluating MAPPO β={beta} ({ckpt})...")
    mappo_cmd = ['python', 'scripts/evaluate.py',
                 '--baseline', 'mappo',
                 '--run_dir', run_dir,
                 '--checkpoint', ckpt_path,
                 '--output', f'{OUTPUT_DIR}/eval_mappo_beta_{tag}.json',
                 '--record',
                 '--record_output', f'{OUTPUT_DIR}/trajectory_mappo_beta_{tag}.pkl'] + SHARED_FLAGS
    subprocess.run(mappo_cmd)

## 🎯 Final Results Summary


In [ ]:
print('=' * 70)
print('📊 FINAL RESULTS — Paper Claims Verification')
print('=' * 70)

# Hard budget
beta0_metrics = OUTPUT_DIR + '/beta_0_0/metrics.jsonl'
if os.path.exists(beta0_metrics):
    with open(beta0_metrics) as f: records = [json.loads(l) for l in f]
    max_beta = max(r['actual_beta'] for r in records)
    min_lam = min(r['lambda'] for r in records)
    print('\n✅ Hard budget (β=0.0):')
    print('   Max actual β across %d updates: %.4f' % (len(records), max_beta))
    print('   Min λ: %.4f (should be ≥ 0.0)' % min_lam)

# Lagrangian stability
print('\n✅ Lagrangian dynamics per β:')
for beta in BETAS:
    tag = str(beta).replace('.', '_')
    path = OUTPUT_DIR + '/beta_' + tag + '/metrics.jsonl'
    if not os.path.exists(path): continue
    with open(path) as f: records = [json.loads(l) for l in f]
    lams = [r['lambda'] for r in records]
    covs = [r['coverage'] for r in records]
    print('   β=%.2f: λ∈[%.3f, %.3f], coverage∈[%.3f, %.3f], final=%.3f' % (
        beta, min(lams), max(lams), min(covs), max(covs), covs[-1]))

# Comparison table
print('\n📊 Coverage Comparison:')
print('%-18s %-12s %-10s %-20s' % ('Policy', 'Coverage', 'T95', 'Bandwidth (bits)'))
print('-' * 60)
eval_files = [('Frontier', OUTPUT_DIR + '/eval_frontier.json')] + \
             [('MAPPO β=' + str(b), OUTPUT_DIR + '/eval_mappo_beta_' + str(b).replace('.', '_') + '.json') for b in BETAS]
for name, path in eval_files:
    if not os.path.exists(path):
        print('%-18s -- (missing)' % name)
        continue
    with open(path) as f: d = json.load(f)
    cov  = d.get('coverage', 0)
    t95  = d.get('t95', -1)
    bw   = d.get('bandwidth_bits', d.get('bandwidth', 0))
    print('%-18s %-12.4f %-10s %-20.1f' % (name, cov, t95, bw))

print('\n' + '=' * 70)
print('📦 All artifacts saved to: ' + OUTPUT_DIR)
print('   - Checkpoints:  beta_*/checkpoints/')
print('   - Metrics:      beta_*/metrics.jsonl')
print('   - Figures:      paper_figures.png')
print('   - Trajectories: trajectory_*.pkl (for Streamlit dashboard)')
print('=' * 70)


## 🎬 Next Steps

1. Download artifacts from `/kaggle/working/experiment_output/`.
2. Run the Streamlit dashboard locally:
   ```bash
   streamlit run app.py
   ```
3. Sanity-check the four figures: coverage curves should slope **up**, λ should stay ≥ 0, and the Pareto scatter should show a monotone trade-off.


In [ ]:
import os, json

OUTPUT_DIR = '/kaggle/working/experiment_output'
BETAS = [0.0, 0.25, 0.5, 0.75, 1.0]

print('=' * 70)
print('📊 FINAL RESULTS — Paper Claims Verification')
print('=' * 70)

# Hard budget
beta0_metrics = f'{OUTPUT_DIR}/beta_0_0/metrics.jsonl'
if os.path.exists(beta0_metrics):
    with open(beta0_metrics) as f: records = [json.loads(l) for l in f]
    max_beta = max(r['actual_beta'] for r in records)
    min_lam = min(r['lambda'] for r in records)
    print(f'\n✅ Hard budget (β=0.0):')
    print(f'   Max actual β across {len(records)} updates: {max_beta:.4f}')
    print(f'   Min λ: {min_lam:.4f} (should be ≥ 0.0)')

# Lagrangian stability
print(f'\n✅ Lagrangian dynamics per β:')
for beta in BETAS:
    tag = str(beta).replace('.', '_')
    path = f'{OUTPUT_DIR}/beta_{tag}/metrics.jsonl'
    if not os.path.exists(path): continue
    with open(path) as f: records = [json.loads(l) for l in f]
    lams = [r['lambda'] for r in records]
    covs = [r['coverage'] for r in records]
    print(f'   β={beta:.2f}: λ∈[{min(lams):.3f}, {max(lams):.3f}], coverage∈[{min(covs):.3f}, {max(covs):.3f}], final={covs[-1]:.3f}')

# Comparison table
print(f'\n📊 Coverage Comparison:')
print(f'{"Policy":<18} {"Coverage":<12} {"T95":<10} {"Bandwidth (bits)":<20}')
print('-' * 60)
eval_files = [('Frontier', f'{OUTPUT_DIR}/eval_frontier.json')] + \
             [(f'MAPPO β={b}', f'{OUTPUT_DIR}/eval_mappo_beta_{str(b).replace(".", "_")}.json') for b in BETAS]
for name, path in eval_files:
    if not os.path.exists(path):
        print(f'{name:<18} -- (missing)')
        continue
    with open(path) as f: d = json.load(f)
    cov  = d.get('coverage', 0)
    t95  = d.get('t95', -1)
    bw   = d.get('bandwidth_bits', d.get('bandwidth', 0))
    print(f'{name:<18} {cov:<12.4f} {str(t95):<10} {bw:<20.1f}')

print('\n' + '=' * 70)
print(f'📦 All artifacts saved to: {OUTPUT_DIR}')
print('   - Checkpoints:  beta_*/checkpoints/')
print('   - Metrics:      beta_*/metrics.jsonl')
print('   - Figures:      paper_figures.png')
print('   - Trajectories: trajectory_*.pkl (for Streamlit dashboard)')
print('=' * 70)